In [ ]:
!pip install mat73

In [ ]:
import re
import mat73
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

In [ ]:
def load_mat_any(path):
    return mat73.loadmat(str(path))

LEADFIELD_STORAGE_DIR = Path.cwd() / "leadfield_calibrated" / "fsavLEADFIELD_4_GEDAI.mat"
M = load_mat_any(LEADFIELD_STORAGE_DIR)

L = M["leadfield4GEDAI"]

electrodes = L["electrodes"]

In [ ]:
# TO USE YOUR OWN CHANNELS CHANGE THIS DATAFRAME.
# EXPECTED channel_name (ORIGINAL CHANNEL NAMES)
# AND X, Y, Z Coordinates. MUST BE DATAFRAME.
MY_CHANNEL_LOCATION = Path.cwd() / "leadfield_calibrated" / "GSN_HydroCel_129.sfp"
chdf = pd.read_csv(
    MY_CHANNEL_LOCATION, 
    sep="\t", 
    header=None, # no header row
    usecols=[0, 1, 2, 3], # only keep columns 1,2,3 (0-based)
    names=["channel_name", "X", "Y", "Z"], # assign names
    dtype={"X": "float64", "Y": "float64", "Z": "float64"})
chdf.head()

In [ ]:
# Load correct montage and map to leadfield matrix channels

# build target set (classic labels only: exclude *h)
tgt = [
    d for d in electrodes
    if d.get("Type") == "EEG"
    and d.get("Loc") is not None
    and not str(d["Name"]).endswith("h")
]
tgt_names = np.array([d["Name"] for d in tgt], dtype=object)
tgt_xyz = np.vstack([np.asarray(d["Loc"], float) for d in tgt])

# build E-names + scale your XYZ to target radius
src = chdf.copy()
src.insert(0, "row", np.arange(len(src))) # row id for sanity (0..N-1)

src_xyz = src[["X","Y","Z"]].to_numpy(float)
src_xyz = src_xyz / np.linalg.norm(src_xyz, axis=1, keepdims=True)
src_xyz = src_xyz * np.median(np.linalg.norm(tgt_xyz, axis=1))

# nearest-neighbor mapping (outputs are in the same order as src_xyz input)
tree = cKDTree(tgt_xyz)
dists, idxs = tree.query(src_xyz, k=1)
src["mapped_name"] = tgt_names[idxs]
src["dist_mm"] = dists * 1000.0

# exact label matches override
tgt_name_set = set(tgt_names.tolist())
mask_exact = src["channel_name"].astype(str).isin(tgt_name_set)
src.loc[mask_exact, "mapped_name"] = src.loc[mask_exact, "channel_name"].astype(str)
src.loc[mask_exact, "dist_mm"] = 0.0

print("median distance (mm):", float(np.median(src["dist_mm"])))
print()  # first 25 rows, same order

selected_channels = src["mapped_name"].tolist() # Contains all mapped names in order
src.loc[:, ["row","channel_name","mapped_name","dist_mm"]].head()

In [ ]:
def _norm(lbl: str) -> str:
    return re.sub(r"[^0-9a-z]+", "", lbl.lower())

def get_leadfield_selected(matdict, selected_channels):
    L = matdict["leadfield4GEDAI"]
    Gain = np.asarray(L["Gain"]) # [channels x sources]
    gram = np.asarray(L["gram_matrix_avref"]) # [channels x channels]
    electrodes = L["electrodes"]

    # extract template labels
    if isinstance(electrodes, (list, tuple)):
        template_labels = [e["Name"] if isinstance(e, dict) else str(e) for e in electrodes]
    elif isinstance(electrodes, dict) and "Name" in electrodes:
        names = electrodes["Name"]
        template_labels = list(names) if isinstance(names, (list, tuple)) else [str(names)]
    else:
        template_labels = list(map(str, electrodes))

    lut = {_norm(t): i for i, t in enumerate(template_labels)}
    idx = np.array([lut.get(_norm(ch), -1) for ch in selected_channels], dtype=int)
    if (idx < 0).any():
        missing = [ch for ch, i in zip(selected_channels, idx) if i < 0]
        raise ValueError(f"Electrode labels not found in template: {missing}")

    Gain_sel = Gain[idx, :]
    gram_sel = gram[np.ix_(idx, idx)]
    labels_sel = [template_labels[i] for i in idx.tolist()]
    return Gain_sel, gram_sel, labels_sel

In [ ]:
# Select channel refCov
Gain_sel, gram_sel, labels_sel = get_leadfield_selected(M, selected_channels)

your_ref_cov = np.asarray(gram_sel, np.float64)
your_ref_cov.shape 
np.save(Path.cwd() / "generated_ref_cov.npy", your_ref_cov)